# Recommender Systems from Scratch

The notebook checks masking, leave-one-out evaluation, and a deterministic cold-start fallback.

In [ ]:
from pathlib import Path
import sys

LESSON_REL = Path('phases/02-ml-fundamentals/19-recommender-systems')
MODULE_FILE = 'main.py'
roots = [Path.cwd(), *Path.cwd().parents]
candidates = [Path.cwd() / LESSON_REL / 'code', Path.cwd() / 'code']
candidates.extend(root / LESSON_REL / 'code' for root in roots)
candidates.extend(root / 'code' for root in roots)
CODE = next((candidate.resolve() for candidate in candidates if (candidate / MODULE_FILE).is_file()), None)
if CODE is None:
    raise RuntimeError('Could not locate ' + str(LESSON_REL / 'code' / MODULE_FILE))
sys.path.insert(0, str(CODE))
sys.modules.pop('main', None)

## Build It

Run the next cell from the lesson directory; all values are local, deterministic fixtures.

In [ ]:
import numpy as np
import main as rec

matrix = np.array([[1, 1, 0, 0], [1, 0, 1, 0], [0, 1, 1, 1]], dtype=float)
assert rec.popularity_scores(matrix).tolist() == [2.0, 2.0, 2.0, 1.0]
train, held_out = rec.leave_one_out(matrix)
ranked = rec.recommend(train, 0, method='neighbors', k=2)
assert all(train[0, item] == 0 for item in ranked)
model = rec.factorize(matrix, factors=2, epochs=5, seed=3)
np.testing.assert_allclose(model.users, rec.factorize(matrix, factors=2, epochs=5, seed=3).users)
assert rec.recall_at_k([2, 3], {2}, 2) == 1.0
try:
    rec.ndcg_at_k([1, 1], {1}, 2)
except ValueError:
    pass
else:
    raise AssertionError('duplicate ranked IDs must be rejected')
cold = np.array([[0, 0, 0], [1, 0, 1]], dtype=float)
fallback = rec.recommend(cold, 0, method='popularity', k=3)
assert rec.recommend(cold, 0, method='factors', k=3) == fallback
try:
    rec.factorize(np.zeros((2, 3)))
except ValueError:
    pass
else:
    raise AssertionError('all-zero factorization must be rejected')

## Exercises and Ship It

Fill the evaluation card with the interaction meaning, held-out item, candidate policy, K, and a popularity cold-start list; add a diversity or safety check before treating ranking metrics as a product result.

Record the observed shape/value and update the lesson output card. A notebook result is evidence for this fixture, not a production guarantee.